# Signal State- Event, Validation, Online State, Residuals

Goal: Build a snall streaming pipeline that in gests noisy events, validates them, maintaines an online estate etimator (EMA), and computes residuals for monitoring.



In [10]:
#Imports, confg, dataclasses
from __future__ import annotations

from dataclasses import dataclass
from datetime import datetime, timedelta, timezone
from typing import Any, Dict, Iterable, List, Optional, Tuple
import math
import random


## Event generation (Synthetic Strean)

We simulate a sensor stream with:
- Gaussian noise
- Missing values (None)
- Outliers (Spikes)
- Occasional offset faults (Persistent Bias)
- A small amount of out-of-order-delivery


In [11]:
@dataclass(frozen=True)
class EventConfig:
    seed: int = 7
    start_time_utc: datetime = datetime(2026, 2, 21, 12, 0, 0, tzinfo=timezone.utc)
    step_seconds: int = 5

    true_signal_base: float = 20.0
    true_signal_drift_per_min: float = 0.02
    noise_std: float = 0.4

    missing_rate: float = 0.03
    outlier_rate: float = 0.02
    fault_rate: float = 0.01

    outlier_magnitude: float = 8.0
    fault_offset: float = 3.0

    sensor_ids: Tuple[str, ...] = ("S1", "S2", "S3")


def _gaussian(rng: random.Random, mean: float, std: float) -> float:
    u1 = max(rng.random(), 1e-12)
    u2 = max(rng.random(), 1e-12)
    z0 = math.sqrt(-2.0 * math.log(u1)) * math.cos(2.0 * math.pi * u2)
    return mean + std * z0


def generate_events(cfg: EventConfig, n: int) -> List[Dict[str, Any]]:
    rng = random.Random(cfg.seed)
    events: List[Dict[str, Any]] = []

    fault_active = {sid: False for sid in cfg.sensor_ids}
    fault_remaining = {sid: 0 for sid in cfg.sensor_ids}

    for i in range(n):
        ts = cfg.start_time_utc + timedelta(seconds=i * cfg.step_seconds)
        minutes = (i * cfg.step_seconds) / 60.0
        true_value = cfg.true_signal_base + cfg.true_signal_drift_per_min * minutes

        for sid in cfg.sensor_ids:
            if (not fault_active[sid]) and (rng.random() < cfg.fault_rate):
                fault_active[sid] = True
                fault_remaining[sid] = rng.randint(6, 20)

            if fault_active[sid]:
                fault_remaining[sid] -= 1
                if fault_remaining[sid] <= 0:
                    fault_active[sid] = False
                
                # missing
            if rng.random() < cfg.missing_rate:
                measured = None
            else:
                measured = _gaussian(rng, true_value, cfg.noise_std)

                # persistent offset fault
                if fault_active[sid]:
                    measured += cfg.fault_offset

                # outlier spike    
                if rng.random() < cfg.outlier_rate:
                    sign = -1.0 if rng.random() < 0.5 else 1.0
                    measured += sign * cfg.outlier_magnitude

            events.append(
                {
                    "event_id": f"{ts.isoformat()}::{sid}",
                    "timestamp": ts.isoformat(),
                    "sensor_id": sid,
                    "value": measured,
                    "units": "C",
                    'fault': fault_active[sid], 
                }
            )
    # small chance of out-of-order events (safe guard)
    if n >= 30:
        j = rng.randint(5, min(len(events) - 1, 50))
        k = rng.randint(0, j - 1)
        events[j], events[k] = events[k], events[j]

    return events

# Validation Boundary

We treat incoming evenys as untrusted:
- Enforce required keys
- Parse ISO timestamps
- Keep missing values as None
- Reject non-finite floats
- Enforce unique event_id

In [12]:
@dataclass(frozen=True)
class ValidatedEvent:
    event_id: str
    timestamp: datetime
    sensor_id: str
    value: Optional[float]
    units: str
    fault: bool


class ValidationError(Exception):
    pass


def validate_events(raw: Iterable[Dict[str, Any]]) -> List[ValidatedEvent]:
    out: List[ValidatedEvent] = []
    seen_ids = set()

    for idx, e in enumerate(raw):
        if not isinstance(e, dict):
            raise ValidationError(f"Event at index {idx} is not a dict")

        for k in ("event_id", "timestamp", "sensor_id", "units"):
            if k not in e:
                raise ValidationError(f"Missing key '{k}' at index {idx}")

        event_id = str(e["event_id"])
        if event_id in seen_ids:
            raise ValidationError(f"Duplicate event_id '{event_id}' at index {idx}")
        seen_ids.add(event_id)

        ts_raw = e["timestamp"]
        try:
            ts = datetime.fromisoformat(str(ts_raw))
            if ts.tzinfo is None:
                ts = ts.replace(tzinfo=timezone.utc)
        except Exception as ex:
            raise ValidationError(f"Bad timestamp '{ts_raw}' at index {idx}") from ex

        sensor_id = str(e["sensor_id"])
        units = str(e["units"])

        v = e.get("value", None)
        if v is None:
            value = None
        else:
            try:
                value = float(v)
            except Exception as ex:
                raise ValidationError(f"Non-numeric value '{v}' at index {idx}") from ex
            if not math.isfinite(value):
                raise ValidationError(f"Non-finite value '{value}' at index {idx}")
        fault = bool(e.get("fault", False))
        out.append(ValidatedEvent(event_id=event_id, timestamp=ts, sensor_id=sensor_id, value=value, units=units, fault=fault,))

    return out


def basic_profile(events: List[ValidatedEvent]) -> Dict[str, Any]:
    total = len(events)
    missing = sum(1 for e in events if e.value is None)
    out_of_order = sum(1 for i in range(1, total) if events[i].timestamp < events[i - 1].timestamp)

    return {
        "total_events": total,
        "missing_values": missing,
        "missing_rate": (missing / total) if total else 0.0,
        "out_of_order_pairs": out_of_order,
        "unique_sensors": len(set(e.sensor_id for e in events)),
    }

## Online State Estimator (EMA)

State per senor:
- initialized flag
- current EMA mean

Update rule:
- If value is None, do not update state
- Otherwise: mean = alpha * x + (1 - alphs) * mean


In [13]:
@dataclass
class EMAState:
    """
    Online exponential moving average state per sensor.
    alpha in (0,1): higher -> reacts faster, lower -> smoother
    """
    alpha: float
    initialized: bool = False
    mean: float = 0.0


def ema_update(state: EMAState, x: Optional[float]) -> EMAState:
    """
    Update EMA with a new measurement x.
    - If x is None (missing), we keep the previous state (no update).
    - On first non-missing measurement, initialize mean to x.
    """
    if x is None:
        return state

    if not state.initialized:
        state.initialized = True
        state.mean = float(x)
        return state

    state.mean = state.alpha * float(x) + (1.0 - state.alpha) * state.mean
    return state


def run_ema(events: List[ValidatedEvent], alpha: float = 0.2) -> Dict[str, List[Dict[str, Any]]]:
    """
    Process events in stream order and return per-sensor traces:
    each row includes timestamp, observed value, and EMA mean.
    """
    # one EMA state per sensor
    states: Dict[str, EMAState] = {}
    traces: Dict[str, List[Dict[str, Any]]] = {}

    # NOTE: we intentionally do not sort yet; we want to see out-of-order effects first.
    for e in events:
        sid = e.sensor_id
        if sid not in states:
            states[sid] = EMAState(alpha=alpha)
            traces[sid] = []

        ema_update(states[sid], e.value)

        traces[sid].append(
            {
                "timestamp": e.timestamp,
                "event_id": e.event_id,
                "value": e.value,
                "ema": states[sid].mean if states[sid].initialized else None,
            }
        )

    return traces

## Residuals, Simple Outlier Flag

Residual = Value - ema

We tracck a streaming scale estimate using an EMA of |residual| ad flag large deviations.

In [14]:
def add_residuals_and_flags(
    traces: Dict[str, List[Dict[str, Any]]],
    z_thresh: float = 4.0,
    min_scale: float = 0.15,
) -> Dict[str, List[Dict[str, Any]]]:
    """
    Add residual = value - ema and a robust-ish outlier flag.

    We estimate scale per sensor online using an EMA of absolute residuals (a cheap proxy for MAD).
    - scale_t = beta*|resid| + (1-beta)*scale_{t-1}
    - z = |resid| / max(scale, min_scale)

    This is intentionally simple + streaming-friendly.
    """
    beta = 0.15  # how quickly scale adapts

    out: Dict[str, List[Dict[str, Any]]] = {}
    for sid, rows in traces.items():
        scale = None
        out_rows = []

        for r in rows:
            x = r["value"]
            mu = r["ema"]

            if x is None or mu is None:
                out_rows.append({**r, "resid": None, "scale": scale, "z": None, "is_outlier": False})
                continue

            resid = float(x) - float(mu)
            abs_resid = abs(resid)

            if scale is None:
                scale = max(abs_resid, min_scale)
            else:
                scale = beta * abs_resid + (1.0 - beta) * scale
                scale = max(scale, min_scale)

            z = abs_resid / scale
            is_outlier = z >= z_thresh

            out_rows.append({**r, "resid": resid, "scale": scale, "z": z, "is_outlier": is_outlier})

        out[sid] = out_rows

    return out


def summarize_flags(enriched: Dict[str, List[Dict[str, Any]]]) -> Dict[str, Any]:
    summary = {}
    for sid, rows in enriched.items():
        n = len(rows)
        n_missing = sum(1 for r in rows if r["value"] is None)
        n_outliers = sum(1 for r in rows if r.get("is_outlier"))
        summary[sid] = {
            "n": n,
            "missing": n_missing,
            "outliers": n_outliers,
            "outlier_rate": (n_outliers / (n - n_missing)) if (n - n_missing) else 0.0,
        }
    return summary


## Run The Pipeline

This executes:
- Generate raw events.
- Validate them into typed records
- Run EMA per sensor
- Compute residuals and outlier flags

In [15]:
cfg = EventConfig(seed=7)
raw = generate_events(cfg, n=120)
events = validate_events(raw)

prof = basic_profile(events)

traces = run_ema(events, alpha=0.2)
enriched = add_residuals_and_flags(traces, z_thresh=4.0)
summary = summarize_flags(enriched)

prof, summary, enriched["S1"][:8]

({'total_events': 360,
  'missing_values': 12,
  'missing_rate': 0.03333333333333333,
  'out_of_order_pairs': 2,
  'unique_sensors': 3},
 {'S1': {'n': 120,
   'missing': 3,
   'outliers': 1,
   'outlier_rate': 0.008547008547008548},
  'S2': {'n': 120, 'missing': 4, 'outliers': 0, 'outlier_rate': 0.0},
  'S3': {'n': 120,
   'missing': 5,
   'outliers': 2,
   'outlier_rate': 0.017391304347826087}},
 [{'timestamp': datetime.datetime(2026, 2, 21, 12, 0, tzinfo=datetime.timezone.utc),
   'event_id': '2026-02-21T12:00:00+00:00::S1',
   'value': 20.332930552920146,
   'ema': 20.332930552920146,
   'resid': 0.0,
   'scale': 0.15,
   'z': 0.0,
   'is_outlier': False},
  {'timestamp': datetime.datetime(2026, 2, 21, 12, 0, 10, tzinfo=datetime.timezone.utc),
   'event_id': '2026-02-21T12:00:10+00:00::S1',
   'value': 20.408507533752058,
   'ema': 20.34804594908653,
   'resid': 0.060461584665528534,
   'scale': 0.15,
   'z': 0.40307723110352356,
   'is_outlier': False},
  {'timestamp': datetime.dat

## Summary

I built a small streaming pipeline that treats incoming events as untrusted.. It validates schema, parses timestamps, enforced unique IDs, and preserves missingnes explicitly.

For each sensor, it maintains an nline exponential moving average as a latent state estimate. beacause the estimator updates incrementally, it works in real-time without batch recomputation.

Finally, it computes residuals (observed minus state) and used a simple streaming scale estimate to flag large deviations as potential anomalies. The result is an end-to-end event-state-monitoring loop that is easy to inspect.